In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from mal_client import MALClient
from anime_data import AnimeDataClient
from anime_recommender import SimilarityRecommender

load_dotenv(PROJECT_ROOT / ".env")

client_id = os.getenv("CLIENT_ID")

In [3]:
# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})

Users data

In [4]:
# users_data = {}
# users_scores = {}

# users = set({'Cresherhsm', 'Mevoll', 'Cheuns', 'Haileytokar', 'dragonenjoyer', 'chenamaty', 'monkeydirene', 'MarnikBe', 'ssafin', 'zClaw_Epic', 'ayumix3', 'ikodrmz', 'heemini', 'I_grV', 'Opelo_Stradyon', 'DanDeku', 'Toaster_toaster', 'SaniLani', 'ArceusComplex', 'MubE', 'SuricateVoador', 'Captn_Cook', 'LILITH_OG', 'DoomSlayer_OG', 'Sturmx', 'AkiAki_Akira', 'kaninhoppning', 'kuzyadam', 'N0rth_5tar', 'KyleAxity_'})
# mal_client = MALClient(client_id)
# for user in users:
#     user_data = mal_client.get_user_data(user)
#     users_data[user] = user_data
#     scores = mal_client.get_scores(user_data)
#     users_scores[user] = scores

In [ ]:
anime_data_client = AnimeDataClient(client_id, cache_file=PROJECT_ROOT / "data" / "anime_cache.json")

In [6]:
anime_data = anime_data_client.get_cache()

Build features

In [7]:
from anime_features import AnimeFeatureBuilder

builder = AnimeFeatureBuilder(
    anime_data,
    max_tfidf_features=3000,
    n_svd_components=300
)

anime_df = builder.build_features()

builder.svd_explained_variance

np.float64(0.37937105892729933)

Convert each anime in df to vectors

In [8]:
recommender = SimilarityRecommender()
anime_vectors = recommender.create_anime_vectors(anime_df)
anime_df_scaled = recommender.anime_df_scaled

Get user Data

In [ ]:
username = "chekkit"
user_client = MALClient(client_id)

user_data = user_client.get_user_data(username)
user_scores = user_client.get_scores(user_data)

Tune SVD components

In [ ]:
from anime_evaluation import HitRateEvaluator, RankingMetricEvaluator
from anime_features import AnimeFeatureBuilder

svd_component_results = []

n_runs = 100
clip_predictions = False
max_features = [2000, 3000, 4000, 5000]
components = [200, 300, 400]
weights_uncertainty = [14]
tuning_top_ks = [5, 10]

for n_feature in max_features:
    for component in components:
        builder = AnimeFeatureBuilder(
            anime_data,
            max_tfidf_features=n_feature,
            n_svd_components=component,
        )

        component_anime_df = builder.build_features()

        recommender = SimilarityRecommender()
        recommender.create_anime_vectors(component_anime_df)
        component_anime_df_scaled = recommender.anime_df_scaled

        hitman = HitRateEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )

        (
            bayesian_results,
            bayesian_summary,
            best_bayesian_weights,
            baseline_results,
            baseline_summary,
        ) = hitman.tune_bayesian_uncertainty(
            weights=weights_uncertainty,
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
            clip_predictions=clip_predictions,
        )

        ranking_evaluator = RankingMetricEvaluator(
            anime_df_scaled=component_anime_df_scaled,
            anime_df=component_anime_df,
            scores=user_scores,
            anime_data_client=anime_data_client,
            anime_data=anime_data,
            builder=builder,
            recommender=recommender,
        )
        _, ranking_summary = ranking_evaluator.tune_bayesian_uncertainty_ranking(
            weights=weights_uncertainty,
            n_runs=n_runs,
            top_ks=tuning_top_ks,
            random_state=42,
            clip_predictions=clip_predictions,
        )

        bayesian_summary = bayesian_summary.rename(
            columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
        )
        ranking_summary = ranking_summary.rename(
            columns={"uncertainty_weight": "bayesian_uncertainty_weight"}
        )

        average_metrics = (
            bayesian_summary
            .merge(
                ranking_summary,
                on=["bayesian_uncertainty_weight", "k"],
                how="left",
            )
            .merge(
                baseline_summary,
                on="k",
                how="left",
            )
        )
        average_metrics["component"] = component
        average_metrics["n_feature"] = n_feature
        average_metrics["clip_predictions"] = clip_predictions
        average_metrics["n_runs"] = n_runs
        average_metrics["svd_explained_variance"] = builder.svd_explained_variance

        svd_component_results.append(average_metrics)

svd_component_summary = (
    pd.concat(svd_component_results, ignore_index=True)
    .sort_values(
        ["k", "avg_precision_at_k", "avg_ndcg_at_k"],
        ascending=[True, False, False],
    )
)

metrics_path = (
    PROJECT_ROOT
    / "metrics"
    / "current_corpus_4793_anime"
    / f"svd_params_tuning_{n_runs}run_ndcg_2026.csv"
)
metrics_path.parent.mkdir(parents=True, exist_ok=True)
svd_component_summary.to_csv(metrics_path, index=False)

print(f"Saved {metrics_path.relative_to(PROJECT_ROOT)}")
svd_component_summary


## Results

This 100-run sweep evaluated the current 4,793-anime corpus with `clip_predictions=False`, `bayesian_uncertainty_weight=14`, and `top_ks=[5, 10]`. Each TF-IDF/SVD configuration was scored with hit-rate metrics plus NDCG/MRR, and the full table is saved to `metrics/current_corpus_4793_anime/svd_params_tuning_100run_ndcg_2026.csv`.

| Read | Components | Max Features | Precision@5 | NDCG@5 | MRR@5 | Precision@10 | NDCG@10 | MRR@10 | Notes |
|---|---:|---:|---:|---:|---:|---:|---:|---:|---|
| Best top-5 precision | 400 | 4000 | 0.5780 | 0.6788 | 0.9620 | 0.3970 | 0.5165 | 0.9620 | Best choice if short recommendation lists matter most. |
| Tied top-5 precision | 400 | 5000 | 0.5780 | 0.6720 | 0.9578 | 0.3920 | 0.5163 | 0.9595 | Same Precision@5 as 400/4000, but weaker ranking tie-breakers. |
| Best top-10 precision | 300 | 4000 | 0.5740 | 0.6722 | 0.9595 | 0.4100 | 0.5175 | 0.9595 | Best Precision@10, still strong at top-5. |
| Best NDCG/MRR balance | 400 | 3000 | 0.5740 | 0.6869 | 0.9683 | 0.4050 | 0.5241 | 0.9683 | Best ranking-quality tie-breaker across both cutoffs. |

Decision: use **400 components / 4000 max features** if optimizing primarily for short top-5 recommendation precision. If the goal is a more balanced ranking-quality default, **400 components / 3000 max features** is the cleaner choice because it is near the precision leaders and wins the NDCG/MRR tie-breakers. The old low-dimensional settings are safely worse on this corpus, so there is no need to retest 50 or 100 components unless the corpus changes substantially.
